In [ ]:
# Cell 1: Basic imports first
import os
import torch
import gc
import psutil
print("Initial imports successful")

# Cell 2: Memory management
def print_memory_usage():
    process = psutil.Process(os.getpid())
    print(f"Current memory usage: {process.memory_info().rss / 1024 / 1024:.2f} MB")

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Memory management functions defined")
clear_memory()
print_memory_usage()

# Cell 3: Model setup
device = torch.device('cpu')
model_path = os.path.join(os.getcwd(), '..', 'narrativesBERT', 'bertopic_model_28_6_40_150_2005_2010.pkl')
print(f"Looking for model at: {os.path.abspath(model_path)}")
print(f"File exists: {os.path.exists(model_path)}")

# Cell 4: Load SentenceTransformer
from sentence_transformers import SentenceTransformer
print("Creating SentenceTransformer...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print_memory_usage()

# Cell 5: Prepare for model loading
import pickle
import io
print("Setting up model loading...")

# Override torch load
original_load = torch.load
def cpu_load(*args, **kwargs):
    kwargs['map_location'] = torch.device('cpu')
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return original_load(*args, **kwargs)
torch.load = cpu_load

# Cell 6: Load model
try:
    print("Starting model load...")
    with open(model_path, 'rb') as f:
        # Read in smaller chunks
        print("Reading file in chunks...")
        buffer = io.BytesIO()
        chunk_size = 1024 * 512  # 512KB chunks
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            buffer.write(chunk)
            print_memory_usage()
            clear_memory()
        
        # Reset buffer position
        buffer.seek(0)
        
        print("Deserializing model...")
        import joblib
        topic_model = joblib.load(buffer)
        
        print("Model loaded, replacing embedding model...")
        topic_model.embedding_model = embedding_model
        
        print("Model ready!")
        print("\nTopic Model Info:")
        print(topic_model.get_topic_info())

except Exception as e:
    print(f"An error occurred: {str(e)}")
    import traceback
    traceback.print_exc()
finally:
    # Cleanup
    if 'original_load' in locals():
        torch.load = original_load
    clear_memory()

In [ ]:
# Cell 1: Check system resources
import os
import psutil

# Check model file size
model_path = os.path.join(os.getcwd(), '..', 'narrativesBERT', 'model', 'bertopic_model_28_6_40_150_2005_2010.pkl')
model_size_gb = os.path.getsize(model_path) / (1024**3)

# Check available system memory
memory = psutil.virtual_memory()
available_gb = memory.available / (1024**3)
total_gb = memory.total / (1024**3)

print(f"Model file size: {model_size_gb:.2f} GB")
print(f"Available RAM: {available_gb:.2f} GB")
print(f"Total RAM: {total_gb:.2f} GB")
print(f"Memory usage: {memory.percent}%")